In [1]:
from datasets import load_from_disk
from pathlib import Path
import sys
current_cwd = Path.cwd()
project_root = current_cwd.parent
if str(project_root) not in sys.path :
    sys.path.insert(0, str(project_root))
DPO_DATA = Path.cwd().parent/"data"/"dpo_dataset"
dataset = load_from_disk(DPO_DATA)

d:\Anaconda\envs\diffusion\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from transformers import AutoTokenizer
import torch
from peft import AutoPeftModelForCausalLM
model_id = project_root/"outputs"/"sft"/"checkpoint-310"
model = AutoPeftModelForCausalLM.from_pretrained(model_id, torch_dtype = torch.bfloat16, device_map = "auto", is_trainable=True)
tokenizer = AutoTokenizer.from_pretrained(model_id)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:02<00:00, 153.00it/s]
Exception in thread Thread-6 (_readerthread):
Traceback (most recent call last):
  File "d:\Anaconda\envs\diffusion\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "d:\Anaconda\envs\diffusion\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "d:\Anaconda\envs\diffusion\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
  File "d:\Anaconda\envs\diffusion\lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb2 in position 7: invalid start byte


In [ ]:
from transformers import TrainerCallback
import torch
import torch.nn.functional as F
import torch.nn.functional as F

def _to_text(example_field):

    if isinstance(example_field, str):
        return example_field
    if isinstance(example_field, list):
        # conversational: [{"role": "...", "content": "..."}]
        parts = []
        for m in example_field:
            if isinstance(m, dict) and "content" in m:
                parts.append(m["content"])
            else:
                parts.append(str(m))
        return "\n".join(parts)
    return str(example_field)


def build_kl_batch(examples, tokenizer, max_length=None):
    prompts = [_to_text(x["prompt"]) for x in examples]
    chosens = [_to_text(x["chosen"]) for x in examples]

    full_texts = [p + c for p, c in zip(prompts, chosens)]

    full_enc = tokenizer(
        full_texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
        add_special_tokens=False,
    )

    prompt_enc = tokenizer(
        prompts,
        padding=False,
        truncation=True,
        max_length=max_length,
        return_tensors=None,
        add_special_tokens=False,
    )

    input_ids = full_enc["input_ids"]
    attention_mask = full_enc["attention_mask"]

    # response_mask: prompt 部分为0，chosen 部分为1，pad为0
    response_mask = torch.zeros_like(input_ids)

    for i, p_ids in enumerate(prompt_enc["input_ids"]):
        prompt_len = len(p_ids)
        seq_len = int(attention_mask[i].sum().item())
        response_mask[i, prompt_len:seq_len] = 1

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "response_mask": response_mask,
    }


@torch.no_grad()
def compute_token_kl_from_raw_examples(
    model,
    ref_model,
    tokenizer,
    examples,
    device,
    max_length=None,
):
    batch = build_kl_batch(examples, tokenizer, max_length=max_length)

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    response_mask = batch["response_mask"].to(device)

    model.eval()
    if ref_model is not None:
        ref_model.eval()

    # policy
    out = model(input_ids=input_ids, attention_mask=attention_mask)
    policy_logits = out.logits[:, :-1, :]

    # reference
    if ref_model is not None:
        ref_out = ref_model(input_ids=input_ids, attention_mask=attention_mask)
    else:
        with model.disable_adapter():
            ref_out = model(input_ids=input_ids, attention_mask=attention_mask)

    ref_logits = ref_out.logits[:, :-1, :]

    policy_log_probs = F.log_softmax(policy_logits, dim=-1)
    ref_log_probs = F.log_softmax(ref_logits, dim=-1)
    policy_probs = policy_log_probs.exp()

    # token-level KL over vocab
    token_kl = (policy_probs * (policy_log_probs - ref_log_probs)).sum(dim=-1)

    mask = response_mask[:, 1:].float()

    total_kl = (token_kl * mask).sum().item()
    total_tokens = mask.sum().item()

    return total_kl / max(total_tokens, 1.0)


class KLLoggerCallback(TrainerCallback):
    def __init__(self, trainer, eval_dataset, tokenizer, num_samples=32, batch_size=4, max_length=None):
        self.trainer = trainer
        self.eval_dataset = eval_dataset.select(range(min(num_samples, len(eval_dataset))))
        self.tokenizer = tokenizer
        self.batch_size = batch_size
        self.max_length = max_length

    def on_evaluate(self, args, state, control, model=None, **kwargs):

        total_kl = 0.0
        total_weight = 0.0

        # 手动分小批，避免一次塞太多OOM
        for start in range(0, len(self.eval_dataset), self.batch_size):
            examples = [
                self.eval_dataset[i]
                for i in range(start, min(start + self.batch_size, len(self.eval_dataset)))
            ]

            kl_value = compute_token_kl_from_raw_examples(
                model=self.trainer.model,
                ref_model=self.trainer.ref_model,
                tokenizer=self.tokenizer,
                examples=examples,
                device=self.trainer.accelerator.device,
                max_length=self.max_length,
            )

            # Here, we simply weight the results by sample size
            total_kl += kl_value * len(examples)
            total_weight += len(examples)

        final_kl = total_kl / max(total_weight, 1.0)
        print(f"[KLCallback] eval/token_kl_chosen = {final_kl}")
        self.trainer.log({"eval/token_kl_chosen": final_kl})

In [11]:
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig
from transformers import EarlyStoppingCallback
import os
import gc

beta_list = [0.4, 0.5]
peft_config = LoraConfig(
        task_type="CAUSAL_LM",
        r = 16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )

for beta_val in beta_list:
    model = AutoPeftModelForCausalLM.from_pretrained(model_id, torch_dtype = torch.bfloat16, device_map = "auto", is_trainable=True)

    output_dir = os.path.join(project_root, "outputs/dpo", f"dpo_beta_{str(beta_val).replace('.', '_')}")

    dpo_config = DPOConfig(
        output_dir = output_dir,
        per_device_train_batch_size = 2,
        per_device_eval_batch_size = 2,
        num_train_epochs = 10,
        gradient_accumulation_steps= 4,
        optim = "adamw_torch",
        learning_rate = 5e-7,
        beta = beta_val,
        weight_decay = 0.05,
        lr_scheduler_type = "cosine",
        eval_strategy="epoch",
        warmup_steps = 30,
        logging_steps=1,
        logging_strategy="steps",
        report_to="tensorboard",
        save_strategy="epoch",       # Must match eval_strategy or be a multiple of it
        load_best_model_at_end=True,  # use early_stopping
        metric_for_best_model="eval_loss", # or eval/rewards
        greater_is_better=False,      # if accuracy , set it to be True
        save_total_limit=1
    )
    dpo_trainer = DPOTrainer(
        model,
        ref_model =None,
        args=dpo_config,
        train_dataset= dataset["train"],
        eval_dataset= dataset["eval"],
        processing_class = tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
        )
    dpo_trainer.add_callback(
        KLLoggerCallback(
            trainer=dpo_trainer,
            eval_dataset=dataset["eval"],
            tokenizer = tokenizer,
            num_samples=32,
            batch_size= dpo_config.per_device_eval_batch_size,
            max_length=256
        )
    )
    dpo_trainer.train()

    del dpo_trainer
    del model
    gc.collect()
    torch.cuda.empty_cache()
    


Loading weights: 100%|██████████| 338/338 [00:01<00:00, 180.80it/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss
1,0.519068,0.565862
2,0.164401,0.388338
3,0.118130,0.290049
4,0.364519,0.233407
5,0.026758,0.191652
6,0.027417,0.170386
7,0.017635,0.163163
8,0.109308,0.153469
9,0.307629,0.164634
10,0.110318,0.156892


[KLCallback] eval/token_kl_chosen = 1.2096291471037672
[KLCallback] eval/token_kl_chosen = 1.247276917094928
[KLCallback] eval/token_kl_chosen = 1.2789889879158602
[KLCallback] eval/token_kl_chosen = 1.306435461318214
[KLCallback] eval/token_kl_chosen = 1.3293265036529738
[KLCallback] eval/token_kl_chosen = 1.340128821422322


d:\Anaconda\envs\diffusion\lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error Server error '500 Internal Server Error' for url 'https://huggingface.co/Qwen/Qwen2.5-1.5B/resolve/main/config.json' (Request ID: Root=1-69d77910-7d501d7f4e1345cf5d74e226;7c5e53c6-51d5-49e4-bb30-dbe36ffabcfc)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500

Internal Error - We're working hard to fix this as soon as possible! - silently ignoring the lookup for the file config.json in Qwen/Qwen2.5-1.5B.
  warnings.warn(
d:\Anaconda\envs\diffusion\lib\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in Qwen/Qwen2.5-1.5B - will assume that the vocabulary was not modified.
  warnings.warn(


[KLCallback] eval/token_kl_chosen = 1.350845558249749


...

[KLCallback] eval/token_kl_chosen = 1.2470033293796947


d:\Anaconda\envs\diffusion\lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error Server error '503 Service Unavailable' for url 'https://huggingface.co/Qwen/Qwen2.5-1.5B/resolve/main/config.json' (Request ID: Root=1-69d77a7b-53c23c5b68694da6551ffe32;086ee420-d446-4bd1-b00b-037f945a1f07)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/503

Internal Error - We're working hard to fix this as soon as possible! - silently ignoring the lookup for the file config.json in Qwen/Qwen2.5-1.5B.
  warnings.warn(
d:\Anaconda\envs\diffusion\lib\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in Qwen/Qwen2.5-1.5B - will assume that the vocabulary was not modified.
  warnings.warn(
d:\Anaconda\envs\diffusion\lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error Server error '503 Service Unavailable' for url 'htt

[KLCallback] eval/token_kl_chosen = 1.2786272598173023


d:\Anaconda\envs\diffusion\lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error Server error '503 Service Unavailable' for url 'https://huggingface.co/Qwen/Qwen2.5-1.5B/resolve/main/config.json' (Request ID: Root=1-69d77ab5-4390b8433f201e9f32dce28a;9cae950f-94d6-4fb7-ac28-e86ee1a9157f)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/503

Internal Error - We're working hard to fix this as soon as possible! - silently ignoring the lookup for the file config.json in Qwen/Qwen2.5-1.5B.
  warnings.warn(
d:\Anaconda\envs\diffusion\lib\site-packages\peft\utils\save_and_load.py:295: UserWarning: Could not find a config file in Qwen/Qwen2.5-1.5B - will assume that the vocabulary was not modified.
  warnings.warn(
d:\Anaconda\envs\diffusion\lib\site-packages\peft\utils\other.py:1394: UserWarning: Unable to fetch remote file due to the following error Server error '503 Service Unavailable' for url 'htt

[KLCallback] eval/token_kl_chosen = 1.3061541251303175
[KLCallback] eval/token_kl_chosen = 1.3204538263563823
[KLCallback] eval/token_kl_chosen = 1.3384461716649407
[KLCallback] eval/token_kl_chosen = 1.344923013459489
[KLCallback] eval/token_kl_chosen = 1.3482999724279021
[KLCallback] eval/token_kl_chosen = 1.3529445172490921
[KLCallback] eval/token_kl_chosen = 1.3542626000783382


In [16]:
## print the log figure, or you can use tensorboard
from src.plot_training_metrics import plot_trainer_state_metrics
import glob

for beta_val in beta_list :
    base_dir = os.path.join(project_root, "outputs/dpo", f"dpo_beta_{str(beta_val).replace('.', '_')}")
    
    checkpoint_folders = glob.glob(os.path.join(base_dir, "checkpoint-*")) # we assume that there is only one checkpoint file
    
    if not checkpoint_folders:
        print(f"⚠️ Warning , cannot find any checkpoint file")
        continue
    
    best_checkpoint_path = checkpoint_folders[0]
    
    trainer_state_path = os.path.join(best_checkpoint_path, "trainer_state.json")
    output_dir = os.path.join(project_root, "outputs/dpo", f"dpo_beta_{str(beta_val).replace('.', '_')}", "metrics")
    plot_trainer_state_metrics(
        trainer_state_path = trainer_state_path,
        output_dir = output_dir,
        run_name="dpo_latest",
    )